In [ ]:
import sys
import pyspark.sql.functions as F
from src.config import data_paths, filter_map
from src.data_ingestion import DataLoader,DataMerger,Preprocessor
from src.logger import logger
from src.exception import M5Exception
from src.utils import spark

import pandas as pd
import numpy as np

In [ ]:
prod_cus_dim_path = data_paths['bronze']['processed_data']['product_customer_dim']
joined_data_path = data_paths['bronze']['processed_data']['joined_data']

In [6]:
data = pd.read_parquet(joined_data_path)
data = data.sort_values(by=['id','yearweek'])
data["date"] = pd.to_datetime(data['yearweek'].astype(str) + "1", format="%G%V%u")
data.head()


,id,yearweek,sales,sell_price,snap_CA_days,snap_TX_days,snap_WI_days,event_flag,f_event_stpatricksday,f_event_martinlutherkingday,...,f_event_mothers_day,f_event_eidaladha,f_event_easter,f_event_superbowl,f_event_chanukah_end,f_event_memorialday,f_event_fathers_day,f_event_ramadan_starts,f_event_valentinesday,date
0,FOODS_1_001_CA_1,201104,3.0,2.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2011-01-24
1,FOODS_1_001_CA_1,201105,7.0,2.0,5,3,3,1,0,0,...,0,0,0,1,0,0,0,0,0,2011-01-31
2,FOODS_1_001_CA_1,201106,7.0,2.0,4,5,4,0,0,0,...,0,0,0,0,0,0,0,0,0,2011-02-07
3,FOODS_1_001_CA_1,201107,6.0,2.0,0,1,2,1,0,0,...,0,0,0,0,0,0,0,0,1,2011-02-14
4,FOODS_1_001_CA_1,201108,6.0,2.0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,2011-02-21


In [7]:
df = data[['id','date','sales']].rename({'id':'unique_id','date':'ds','sales':'y'})
df.head()

,id,date,sales
0,FOODS_1_001_CA_1,2011-01-24,3.0
1,FOODS_1_001_CA_1,2011-01-31,7.0
2,FOODS_1_001_CA_1,2011-02-07,7.0
3,FOODS_1_001_CA_1,2011-02-14,6.0
4,FOODS_1_001_CA_1,2011-02-21,6.0


In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive

c:\m5_forecasting\m5-demand-forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: DLL load failed while importing _lib: An Application Control policy has blocked this file.

In [ ]:
models = [
    Naive(),
    SeasonalNaive(season_length=52),
    RandomWalkWithDrift()
]